# Start here: uv basics

This is the first notebook of the course, so it does two jobs. It explains how the course works and what each module teaches, and it teaches uv, the tool that manages the Python version, the packages, and the environment for every module after this one. By the end you'll have practiced every common uv command on a throwaway project, added the packages later modules need, confirmed that every download stays inside the `ML_Lab` folder, and checked that PyTorch and MLX can use your Mac's GPU.

**In this notebook**

1. How this course works: setup, habits, the roadmap, and what to watch alongside
2. The problem uv solves
3. The three files that matter
4. The project layout
5. Commands you'll actually use
6. Practice on a throwaway project
7. Add the packages this course needs
8. Where your downloads live
9. Run tools without installing them
10. Check your hardware stack
11. Commit your work
12. If a package won't install

**Before you start**

- Prerequisites: none. This is module 00.
- Open the whole `ML_Lab` folder in VS Code (File > Open Folder), not just this file, and pick the `.venv` kernel inside `ML_Lab` in the top right. Section 1 walks through it.
- No apps need to be running, but you need an internet connection. Section 7 adds `openai`, `matplotlib`, and `ruff` to the project (a few tens of MB), section 9 downloads a tiny tool, and exercise 2 uses Python 3.12, which uv downloads if your Mac doesn't have it (that one lands in uv's own Python folder in your home directory, outside `ML_Lab`, as section 8 explains).
- Time: about 50 minutes.
- Downloads stay inside `ML_Lab`. If you cloned the repo fresh, run `uv run python -m ml_lab setup` once before section 6, so uv's own downloads land there from the start. You can do that in iTerm2 from `ML_Lab`, or jump ahead and run the setup cell in section 8. Section 8 explains what it does and checks it. If you've already run it, skip this.
- Section 6 creates a scratch folder, `practice_app/`, next to this notebook. It's for practice only and stays out of git.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
import tomllib
from pathlib import Path

# Notebooks run with their own folder as the working directory, so the project root is two levels up.
HERE = Path.cwd()
ROOT = (HERE / ".." / "..").resolve()
PRACTICE = HERE / "practice_app"   # the throwaway project from section 6

# The ! cells run uv in a pseudo-terminal, where it draws progress spinners that clutter the output.
os.environ["UV_NO_PROGRESS"] = "1"

# Section 6 deletes and recreates practice_app/, so refuse to run anywhere except lessons/00_uv.
assert HERE.name == "00_uv" and (ROOT / "pyproject.toml").exists(), (
    f"Expected to run from lessons/00_uv inside ML_Lab, but the working directory is {HERE}"
)

print("Notebook folder:", HERE)
print("Project root:   ", ROOT)
print("uv found at:    ", shutil.which("uv") or "NOT FOUND (see the note below)")
!uv --version

A line that starts with `!` runs as a shell command, exactly as if you typed it into iTerm2 from this notebook's folder. Most of this notebook is `!uv ...` cells.

If uv shows `NOT FOUND`, VS Code didn't pick up your shell's `PATH` (where Homebrew put uv). Quit VS Code fully with Cmd+Q and reopen it. If that doesn't fix it, open VS Code from iTerm2 with `code ~/Developer/ML_Lab` (install the `code` command first: Command Palette > "Shell Command: Install 'code' command in PATH").

## 1. How this course works

The course is ten modules in `ML_Lab/lessons/`, one folder each, and each module is a single notebook like this one. Everything you need to learn is in the notebooks. Work through them in order, because each one builds on the ones before it.

Set up VS Code once:

1. Open VS Code, choose File > Open Folder, and pick `~/Developer/ML_Lab`. Opening the whole folder lets VS Code find the project's `.venv`.
2. Open a module's notebook from the Explorer sidebar, for example `lessons/00_uv/uv_basics.ipynb`.
3. Click "Select Kernel" in the top right, choose "Python Environments", and pick the one whose path ends in `ML_Lab/.venv/bin/python`. VS Code remembers the choice for each notebook.
4. Run a cell with Shift+Enter. It runs the cell and moves to the next one.

Habits that make the course work:

- Before you run a cell that prints a shape, a number, or a file, predict what it will print. The cells where you guess wrong teach you the most.
- Do the exercises before you look at the solutions at the bottom of each notebook. The struggle is where the learning happens.
- Commit after each module. From iTerm2 inside `ML_Lab`, that's `git add . && git commit -m "Finish module 02"` (with the right module number). Section 11 does this module's commit from the notebook.
- Put your own experiments in `ML_Lab/experiments/` (create it when you need it), not inside `lessons/`, so the course stays clean and your work stays yours.

### The roadmap

| # | Module | What you'll be able to do | Notebook (inside `lessons/`) |
|---|---|---|---|
| 00 | uv | Manage Python projects, dependencies, and environments without breaking anything | `00_uv/uv_basics.ipynb` |
| 01 | LM Studio | Run open-weight LLMs locally, understand quantization, call a local model from Python | `01_lm_studio/lm_studio_api.ipynb` |
| 02 | PyTorch basics | Work with tensors, use your Mac's GPU, understand autograd, gradient descent, and optimizers like Adam, write a training loop | `02_pytorch_basics/pytorch_basics.ipynb` |
| 03 | micrograd | Build backpropagation from scratch, redo it with matrices, and understand what `loss.backward()` actually does | `03_micrograd/micrograd.ipynb` |
| 04 | Hugging Face | Pull models and datasets from the Hub, use tokenizers and `transformers`, push your own models | `04_hugging_face/hugging_face.ipynb` |
| 05 | Training + W&B | Train a real network on a real dataset, keep its gradients healthy, and track every run in Weights & Biases | `05_training_and_wandb/training_with_wandb.ipynb` |
| 06 | GPT from scratch | Build and train a small transformer language model end to end | `06_gpt_from_scratch/gpt_from_scratch.ipynb` |
| 07 | MLX + LoRA | Fine-tune a real open-weight LLM on your Mac with your own data | `07_mlx_lora/lora_finetuning.ipynb` |
| 08 | Serving | Serve your fine-tuned model locally, call it from a FastAPI backend, and compare hosting costs | `08_serving/serving.ipynb` |
| 09 | Modal | Run functions on rented cloud GPUs by the second, and keep the cost under control | `09_modal/modal_cloud_gpus.ipynb` |

### Why there's no TensorFlow module

PyTorch is the default for research and for nearly all LLM work. Most papers, Hugging Face models, and open training code (nanoGPT, Llama fine-tuning, LoRA tooling) are PyTorch first. You already use JAX at the Hume Center, and MLX covers Apple Silicon. TensorFlow support on Mac GPUs is also weak. If a class or job ever requires TensorFlow, the concepts from module 02 carry over and you'll pick up the API in a day.

### What to watch alongside

- Andrej Karpathy, "Neural Networks: Zero to Hero" (YouTube playlist). Watch "The spelled-out intro to neural networks and backpropagation: building micrograd" with module 03. "Building makemore Part 4: Becoming a Backprop Ninja" pairs with module 03 section 8 (backprop with matrices), and "Building makemore Part 3: Activations & Gradients, BatchNorm" pairs with module 05 section 4 (initialization and healthy gradients). Module 06 follows "Let's build GPT: from scratch, in code, spelled out", and "Let's build the GPT Tokenizer" goes deeper on the BPE exercise at the end of module 06.
- 3Blue1Brown, "Essence of Linear Algebra" and "Neural Networks" (YouTube). Worth a weekend before module 02 if your linear algebra feels rusty. The later "Neural Networks" chapters on transformers and attention are a good warm-up for module 06.
- Stanford CS336, "Language Modeling from Scratch" (public lectures). Good for depth after module 06.

## 2. The problem uv solves

If you `pip install` everything into one global Python, projects end up fighting. An app like Darvis wants one version of a library, a class project wants another, and upgrading one breaks the other. The fix is a virtual environment: a private folder of packages for each project, tied to one Python interpreter.

uv manages three things for you: which Python version a project uses, which packages it depends on, and the virtual environment those packages get installed into. It replaces a pile of older tools (pip, pip-tools, virtualenv, pyenv, pipx, Poetry) with one fast binary written in Rust by Astral, the company behind the `ruff` linter. It's 10 to 100 times faster than pip, mostly because it downloads in parallel and keeps a package cache it can reuse instead of downloading the same package twice.

Start with the first problem: which Python? This lists every interpreter uv can find on your Mac.

In [ ]:
!uv python list --only-installed

You probably have several: Homebrew's, maybe one from python.org, maybe some uv downloaded itself. Without a tool that decides per project, `python3` in a terminal means whichever one comes first on your `PATH`, and that can change when you install something unrelated. uv answers the question per project, from a small file you'll see in the next section.

## 3. The three files that matter

Open the `ML_Lab` folder in the VS Code Explorer and find these three.

`pyproject.toml` is what you asked for. It lists your direct dependencies with minimum versions (`torch>=2.15.0`). You change it through `uv add` and `uv remove`, rarely by hand.

`uv.lock` is what you actually got. It records the exact version of every package, including the dependencies of your dependencies, plus a hash of every file so a tampered download refuses to install. Commit this file. It's what lets someone else (or you on a new laptop) recreate the identical environment.

`.venv/` is the installed environment itself: a link to a Python interpreter plus a `site-packages` folder with every package unpacked. Never commit it (it's in `.gitignore`). It's fully rebuildable from `uv.lock`, so if it ever gets into a weird state, delete it and run `uv sync`. Section 6 shows how to do that safely. One catch: section 8's startup file lives inside `.venv`, so after a rebuild you run `uv run python -m ml_lab setup` once more.

Mental model: `pyproject.toml` is the recipe, `uv.lock` is the exact shopping receipt, `.venv` is the cooked meal.

A fourth file shows up in section 8: `uv.toml`, uv's own settings for this Mac (where to keep its package cache). It holds a full path on your machine, so it stays out of git.

Start with the recipe. Python 3.11 and newer can read TOML files with the built-in `tomllib` module.

In [ ]:
pyproject = tomllib.loads((ROOT / "pyproject.toml").read_text())
project = pyproject["project"]

print("name:           ", project["name"])
print("requires-python:", project["requires-python"])
print(".python-version:", (ROOT / ".python-version").read_text().strip())
print("dependencies:")
for dep in project["dependencies"]:
    print("   ", dep)

`requires-python = ">=3.14"` is a promise about which Pythons the project supports. `.python-version` is the answer to section 2's question: the exact interpreter uv uses when it builds `.venv`. Each dependency is a name plus a constraint. `>=` sets a minimum and leaves the exact version to the lock file.

Now the receipt. Predict before you run: you list six packages, so roughly how many does the lock pin?

In [ ]:
lock_path = ROOT / "uv.lock"
lock = tomllib.loads(lock_path.read_text())
packages = lock["package"]
print(f"uv.lock is {lock_path.stat().st_size / 1024:.0f} KB and pins {len(packages)} packages")

torch_entry = next(p for p in packages if p["name"] == "torch")
wheel = torch_entry["wheels"][0]
print(f"\ntorch is pinned to {torch_entry['version']} with {len(torch_entry['wheels'])} prebuilt files (wheels) listed, one per platform and Python version")
print("one of them:", wheel["url"].rsplit("/", 1)[-1])
print("its hash:   ", wheel["hash"][:40] + "...")

nvidia = [p["name"] for p in packages if p["name"].startswith("nvidia")]
installed = list((ROOT / ".venv").glob("lib/python*/site-packages/*.dist-info"))
print(f"\nNVIDIA CUDA packages in the lock: {len(nvidia)}, for example {nvidia[:3]}")
print(f"Packages actually installed in ML_Lab/.venv: {len(installed)}")

What to notice:

- Six direct dependencies became over a hundred pinned packages, because each dependency brings its own dependencies, and those bring theirs. Section 7 shows torch's share.
- The lock is universal. uv solves your dependencies once for every platform and Python version your project allows, so the same `uv.lock` works on your Mac, a Linux GPU box, and a teammate's Windows laptop. That's why it lists NVIDIA's CUDA libraries (torch needs them on Linux) and a separate torch wheel for each platform and Python version. Your Mac installs only its own slice, which is why `.venv` holds fewer packages than the lock pins.
- The hash means a package can't silently change under you. If PyPI ever served different bytes for that exact file, uv would refuse to install it.

Last, the cooked meal. This kernel itself runs from `.venv`:

In [ ]:
print("This kernel's Python:", sys.executable)
print("which links to:      ", Path(sys.executable).resolve())
print("Its environment:     ", sys.prefix)
print("Is it ML_Lab's .venv?", Path(sys.prefix).resolve() == (ROOT / ".venv").resolve())

If the last line says `False`, you picked the wrong kernel. Go back to section 1, step 3.

Notice that `.venv/bin/python` is only a link to an interpreter that lives elsewhere (Homebrew's, or one uv downloaded). A virtual environment doesn't copy Python. It borrows an interpreter and adds its own private `site-packages` folder.

## 4. The project layout

You created ML_Lab with uv's package layout, so there's a `src/ml_lab/` folder and a `[project.scripts]` entry in `pyproject.toml`. That entry means `uv run ml-lab` calls the `main()` function in `src/ml_lab/__init__.py`.

`uv sync` also installs the project itself into `.venv` in editable mode: the environment points at your `src/` folder instead of holding a copy. So anything you put in `src/ml_lab/` can be imported in any notebook with `from ml_lab import ...`. Later it's how you'd turn reusable code (a data loader, a model class) into a proper package. The course already uses it once: `src/ml_lab/cache.py` is what keeps your downloads inside `ML_Lab`, and `src/ml_lab/__main__.py` makes `uv run python -m ml_lab` work (section 8).

In [ ]:
print("[project.scripts]:", pyproject["project"]["scripts"])
print()
print((ROOT / "src" / "ml_lab" / "__init__.py").read_text())

In [ ]:
!uv run ml-lab

In [ ]:
import ml_lab
from ml_lab import main

main()
print("imported from:", ml_lab.__file__)

What to notice: the import path points into `src/`, not into `.venv`. That's the editable install. One catch: Python caches a module the first time you import it, so if you edit a file in `src/ml_lab/` while a notebook is open, restart the kernel (or run `%load_ext autoreload` and `%autoreload 2` once) to pick up the change.

## 5. Commands you'll actually use

| Command | What it does |
|---|---|
| `uv add <pkg>` | Adds a dependency to `pyproject.toml`, updates `uv.lock`, installs it into `.venv` |
| `uv remove <pkg>` | The reverse |
| `uv add --dev <pkg>` | Adds a tool you need while developing (linters, test runners) that your code doesn't import |
| `uv sync` | Makes `.venv` match `uv.lock` exactly. Run it after `git pull` or after deleting `.venv` |
| `uv run <cmd>` | Runs a command inside the environment, syncing first if needed. No activating required |
| `uv tree` | Shows the full dependency tree |
| `uv lock --upgrade` | Bumps every package to the newest version your `pyproject.toml` allows |
| `uv lock --upgrade-package <pkg>` | Bumps just one package |
| `uv python list` | Shows Python versions installed or available |
| `uv python pin 3.12` | Switches the project's Python version (writes `.python-version`) |
| `uvx <tool>` | Runs a command-line tool in a throwaway environment without adding it to your project |
| `uv cache dir` | Prints where uv keeps the packages it downloads. In this repo that's `ML_Lab/.cache/uv` (section 8) |
| `uv cache clean` | Empties that cache. Installed environments keep working, and the next install downloads again |

You'll see people run `source .venv/bin/activate` and then plain `python`. That works too, but `uv run` is safer: it checks that `.venv` matches `uv.lock` and syncs first if it doesn't, so you can't accidentally run against a stale environment after a `git pull`. It also finds the project from any subfolder, which is why the `!uv` cells in this notebook work from `lessons/00_uv`, two folders below the project root.

Tutorials everywhere say `pip install x`. In this repo, translate that to `uv add x`, never `pip install`. A plain `pip install` drops a package into `.venv` without recording it in `pyproject.toml` or `uv.lock`, so nobody else gets it, and the next `uv sync` removes it (sync makes `.venv` match the lock exactly).

## 6. Practice on a throwaway project

Commands that change or delete things are safer to learn on a project you don't care about. This section creates `practice_app/` next to this notebook and puts it through the whole cycle: create, add, run, remove, dev tools, inspect, rebuild. Rerunning the first cell wipes it and starts over, so experiment freely.

The `uv init` flags:

- `--app` makes an application (something you run) rather than a library (something other projects import). In uv 0.13 an app gets the same `src/` package layout ML_Lab has.
- `--no-workspace` matters here. Without it, uv notices it's being run inside another project (ML_Lab) and registers the new one as a member of a workspace, which edits ML_Lab's `pyproject.toml`. Workspaces are how a monorepo keeps several related packages on one shared lock file. Useful later, not what you want for scratch work.
- `--vcs none` tells uv not to set up git for the new project, since ML_Lab's repo already covers this folder.

In [ ]:
# Start fresh every time this section runs. The assert in the setup cell confirmed we're in lessons/00_uv.
shutil.rmtree(PRACTICE, ignore_errors=True)
!uv init --app --no-workspace --vcs none practice_app

In [ ]:
for path in sorted(PRACTICE.rglob("*")):
    if path.is_file():
        print(path.relative_to(HERE))
print()
print((PRACTICE / "pyproject.toml").read_text())

There's a `pyproject.toml` with an empty `dependencies` list, a `.python-version`, and a package in `src/practice_app/`. There's no `.venv` and no `uv.lock` yet. uv creates both the first time a command needs them. Run the app's entry point and look again:

In [ ]:
!cd practice_app && uv run practice-app
print(sorted(p.name for p in PRACTICE.iterdir()))

That one `uv run` resolved the (empty) dependency list, wrote `uv.lock`, created `.venv`, installed the project into it, and ran `main()`.

If you see `warning: VIRTUAL_ENV=... does not match the project environment path .venv and will be ignored`, that's expected. VS Code activated ML_Lab's `.venv` for this kernel, and uv is telling you it correctly uses practice_app's own `.venv` instead.

Now add a package, use it, and remove it. `rich` prints colored text and tables in a terminal. Watch the `+` and `-` lines: they list exactly which packages were installed and removed.

In [ ]:
!cd practice_app && uv add rich

In [ ]:
!cd practice_app && uv run python -c "from rich import print; print('[bold green]it works[/]')"
print(tomllib.loads((PRACTICE / "pyproject.toml").read_text())["project"]["dependencies"])

In [ ]:
!cd practice_app && uv remove rich
print(tomllib.loads((PRACTICE / "pyproject.toml").read_text())["project"]["dependencies"])

What to notice: adding `rich` installed more than one package, because rich has dependencies of its own. Removing it took those out too, since nothing else needed them. You never had to track the extras yourself.

Next, a dev tool. `ruff` is a linter (it finds mistakes and unused code) and a formatter. Your code never imports it, so it belongs in the dev group. The cell writes a file with two unused imports so ruff has something to find.

In [ ]:
messy = PRACTICE / "src" / "practice_app" / "messy.py"
messy.write_text('import os\nimport sys\n\n\ndef greet(name):\n    return "hi " + name\n')

!cd practice_app && uv add --dev ruff
!cd practice_app && uv run ruff check

In [ ]:
!cd practice_app && uv run ruff check --fix
print(messy.read_text())
print((PRACTICE / "pyproject.toml").read_text().split("[build-system]")[1])

`ruff check` flagged both imports as unused (rule `F401`), and `--fix` deleted them. In `pyproject.toml`, ruff went into a separate `[dependency-groups]` table under `dev`, not into `dependencies`. You'll see why that matters in section 7.

`uv tree` shows the whole picture:

In [ ]:
!cd practice_app && uv tree

Last, prove the environment is reproducible: delete `.venv` and rebuild it from `uv.lock`.

In [ ]:
shutil.rmtree(PRACTICE / ".venv")
print("practice_app/.venv exists:", (PRACTICE / ".venv").exists())

start = time.perf_counter()
result = subprocess.run(["uv", "sync"], cwd=PRACTICE, capture_output=True, text=True)
elapsed = time.perf_counter() - start

print(result.stderr)
print(f"Rebuilt practice_app/.venv from uv.lock in {elapsed:.2f} s")

# uv finds ML_Lab/uv.toml by looking in parent folders, so practice_app shares ML_Lab's package cache.
!cd practice_app && uv cache dir

What to notice: the rebuild takes a fraction of a second. practice_app is tiny, but the same command rebuilds ML_Lab's entire environment, torch included, in seconds once the cache holds the packages. The reason is uv's package cache, whose path the last line above prints. Every package uv has downloaded stays there, and installing means cloning files out of the cache (macOS can copy a file instantly without duplicating it on disk) rather than downloading again. That speed is a big part of why so many Python teams have moved to uv.

The cache only helps after it has the packages. Section 8 points uv at a cache inside `ML_Lab`, so if you set that up after you had already installed everything, the first sync into the new, empty cache downloads it all once more. Later syncs are fast again.

Don't try this on ML_Lab from inside a notebook. This kernel's Python lives in `ML_Lab/.venv`, so deleting that folder pulls the floor out from under the running kernel. If you ever want to rebuild ML_Lab's environment from scratch:

1. Shut the notebook kernel down (or quit VS Code).
2. In iTerm2 (the last line puts back the startup file from section 8, which lived inside the `.venv` you just deleted):

```bash
cd ~/Developer/ML_Lab
rm -rf .venv
time uv sync
uv run python -m ml_lab setup
```

3. Reopen the notebook and reselect the `.venv` kernel.

## 7. Add the packages this course needs

Now the real thing, on ML_Lab. Module 01 needs `openai`, the official client that works with any OpenAI-compatible server (including LM Studio and Ollama), and later modules plot with `matplotlib`. `ruff` goes in as a dev dependency, the same as in practice_app.

These cells run from `lessons/00_uv`, and uv walks up the folders until it finds ML_Lab's `pyproject.toml`.

In [ ]:
!uv add openai matplotlib
!uv add --dev ruff

In [ ]:
!git -C ../.. diff pyproject.toml
!git -C ../.. diff --stat

What to notice:

- `dependencies` gained `openai` and `matplotlib`, each with a `>=` minimum set to the version uv just resolved.
- A new `[dependency-groups]` table holds `ruff` under `dev`. `uv sync` installs the dev group on your machine by default. A production build (a Docker image for an API, for example) runs `uv sync --no-dev` and skips it, so linters and test runners never ship to a server.
- `uv.lock` changed by far more lines than `pyproject.toml`, because each new package brought its own dependencies. (`git diff --stat` also lists any other file in the repo you've changed, such as a lesson notebook you ran.)

**Restart the kernel now.** The packages are installed, but a running Python process doesn't reliably see packages that appeared after it started, and a package that was already imported stays at its old version until a restart. Click Restart in the toolbar at the top of the notebook, run the setup cell at the top of this notebook again (it defines `ROOT`, `PRACTICE`, and the imports), then continue with the next cell.

In [ ]:
import matplotlib
import openai

print("openai    ", openai.__version__)
print("matplotlib", matplotlib.__version__)

Now answer the question from section 3: why does the lock pin so many more packages than you asked for? `uv tree` shows it. `--depth 1` limits the tree to your direct dependencies, and `--package torch` shows just torch's subtree.

In [ ]:
!uv tree --depth 1

In [ ]:
!uv tree --package torch

In [ ]:
!uv tree --package torch --depth 1 --universal

What to notice:

- `--depth 1` matches your `pyproject.toml` list, now with exact versions from the lock and `ruff` marked `(group: dev)`.
- On your Mac, torch pulls in a handful of packages (`sympy`, `networkx`, `jinja2`, and others), and some of those pull in more. Multiply that by every direct dependency and you get the size of `uv.lock`.
- `uv tree` normally shows only what your platform installs. `--universal` shows the whole lock, and torch's list grows with packages it needs only on Linux: NVIDIA's CUDA libraries and `triton`, a compiler for GPU code. That's the rest of the answer: the lock carries every platform's dependencies, and your Mac installs only its own.

## 8. Where your downloads live

Everything in this course downloads something: Hugging Face models and datasets (several gigabytes by module 07), PyTorch and Weights & Biases caches, and uv's own package cache. By default each of those tools writes to a hidden folder in your home directory, such as `~/.cache/huggingface` and `~/.cache/uv`. The files end up spread across your Mac, and deleting `ML_Lab` would leave all of them behind.

This repo changes the defaults. Every download goes into `ML_Lab/.cache`, so deleting `ML_Lab` removes the whole course and nothing from it leaks into the rest of your machine. `.cache/` is listed in `.gitignore`, so none of it can reach GitHub.

What lives inside `ML_Lab`:

| What | Where |
|---|---|
| Python packages (torch, mlx, openai, and so on) | `ML_Lab/.venv` |
| Hugging Face models and datasets | `ML_Lab/.cache/huggingface` |
| uv's package cache | `ML_Lab/.cache/uv` |
| Weights & Biases caches | `ML_Lab/.cache/wandb` |
| Anything PyTorch downloads on its own (for example pretrained weights through `torch.hub`) | `ML_Lab/.cache/torch` |
| Each lesson's own outputs: `data/`, `checkpoints/`, `adapters/`, `models/`, and W&B run logs in `wandb/` | Inside that lesson's folder, for example `lessons/07_mlx_lora/adapters/` |

What stays outside, on purpose:

| What | Where | Notes |
|---|---|---|
| Hugging Face login (your token) | `~/.cache/huggingface/token` | It's a credential that your other projects use too, so it stays where the tool put it. |
| W&B login | `~/.netrc` | Same reason. |
| LM Studio's models | `~/.lmstudio/models` | LM Studio manages this folder itself. To move it, open My Models in LM Studio and change the models directory. |
| Ollama's models | `~/.ollama/models` | Ollama reads the `OLLAMA_MODELS` environment variable. For the Ollama app on a Mac, set it with `launchctl setenv OLLAMA_MODELS /your/folder` and restart the app. A variable exported in your shell only reaches `ollama serve` runs from that shell. |

You don't need to move the last two. Three more things stay outside. The first is system-wide installs such as Homebrew. The second is the Python interpreters that uv downloads for you (`uv python dir` prints the folder, and a download only happens if you ask for a Python you don't have, as exercise 2 does on a Mac without Python 3.12). The third is uv's tool commands: `uv tool install` keeps its tools in `uv tool dir`, and a bare `uvx` typed in iTerm2 uses uv's default cache in your home folder. Section 9 shows how the notebook keeps `uvx` inside `ML_Lab` anyway.

### How it works

The tools fall into two groups, so there are two pieces.

Hugging Face, PyTorch, and W&B are Python libraries, and each one reads an environment variable (`HF_HOME`, `TORCH_HOME`, `WANDB_CACHE_DIR`, and a few more) to decide where its files go. `src/ml_lab/cache.py` sets those variables to folders inside `ML_Lab/.cache`. To make that happen in every Python process you start from this project (notebook kernels, `uv run`, scripts, and the subprocesses that module 07 launches for training), setup writes a one-line startup file called `zz_ml_lab_cache.pth` into `.venv`'s `site-packages` folder. When Python starts, it runs any line in a `.pth` file that begins with `import`, so that one line calls `ml_lab.cache.configure()` before your code runs. If you've exported one of these variables yourself, yours wins. The first code cell of the notebooks that download things (modules 04, 05, and 07) also calls `ml_lab.cache.configure()`, so their downloads stay inside `ML_Lab` even if you skipped setup. Setup still matters for everything else: terminals, `uv run` scripts, and your own experiments.

The usual hook for this is a `sitecustomize.py` file, but Homebrew's Python ships its own, and Python would load that one instead of ours. A `.pth` file has no such conflict.

uv is a separate program, not Python code running inside `.venv`, so the startup file can't reach it. uv reads its settings from a `uv.toml` file, which it looks for in the current folder and then in each parent folder. Setup writes `ML_Lab/uv.toml` with one setting, `cache-dir`, pointing at `ML_Lab/.cache/uv`. Any uv command you run anywhere inside `ML_Lab` finds it, including the ones in `practice_app/`. The one exception is `uvx` and the `uv tool` commands: uv counts those as user-level and ignores project settings for them (section 9).

Setup writes that path in full (`/Users/you/Developer/ML_Lab/.cache/uv`) instead of a short `.cache/uv`, because uv resolves a relative `cache-dir` against the folder you run the command from, not against the project root. A short path would quietly create a new cache inside every folder you ran uv from. A full path belongs to one machine and one location, which is why `uv.toml` is git-ignored and has to be regenerated when either changes.

### Check your setup

The status command prints where each tool will put its files. Its first two lines say whether setup has been done on this Mac.

In [ ]:
!uv run python -m ml_lab

`startup file installed` and `uv.toml present` should both say `yes`. Every other line should say `inside ML_Lab`, except the Hugging Face token path, which stays in your home folder. This command calls `configure()` itself, so those lines look right even when the startup file is missing. The two `yes` lines are the ones to trust.

The next cell is the real test. It asks this notebook's own Python what it sees. `huggingface_hub.constants.HF_HUB_CACHE` is the folder the Hugging Face library will download models into.

In [ ]:
from huggingface_hub import constants

values = {"HF_HOME": os.environ.get("HF_HOME"), "HF_HUB_CACHE": constants.HF_HUB_CACHE}
inside = {}
for name, value in values.items():
    inside[name] = bool(value) and Path(value).is_absolute() and Path(value).resolve().is_relative_to(ROOT)
    print(f"{name:13}{value}   {'inside ML_Lab' if inside[name] else 'OUTSIDE ML_Lab'}")

if all(inside.values()):
    print("\nGood: models and datasets downloaded from this notebook land inside ML_Lab/.cache.")
else:
    print(
        "\nThis kernel is NOT using ML_Lab/.cache yet. Run the setup cell below, then restart this kernel "
        "(Restart in the toolbar), rerun the first cell at the top of the notebook, and run this cell again."
    )

Here are the two files setup writes. The startup line wraps its call in `try` and `except`, so a broken checkout never prints errors every time Python starts.

In [ ]:
import sysconfig

startup_file = Path(sysconfig.get_paths()["purelib"]) / "zz_ml_lab_cache.pth"
for path in (startup_file, ROOT / "uv.toml"):
    print(f"--- {os.path.relpath(path, ROOT)}")
    print(path.read_text() if path.exists() else "(missing, run the setup cell below)")

### Run setup

`uv run python -m ml_lab setup` writes the startup file into `.venv` and writes `uv.toml`. It only creates or overwrites those two small files (and an empty `.cache/` folder if it's missing), so it's safe to rerun as often as you like. If the checks above passed, you can skip it. If they didn't, run it now:

In [ ]:
# Safe to rerun: it rewrites the same two small files each time.
!uv run python -m ml_lab setup

A running kernel keeps the environment it started with, so restart this notebook's kernel after setup, rerun the first cell at the top, and rerun the check cell above. Anything you start later (a new kernel, a new iTerm2 tab running `uv run`) picks the settings up on its own.

On a new machine, the whole sequence is:

```bash
git clone https://github.com/pujanpatel08/ml_lab.git ~/Developer/ML_Lab
cd ~/Developer/ML_Lab
uv sync
uv run python -m ml_lab setup
```

Run setup again in three situations, and only these:

1. After a fresh clone, because nothing has run it yet.
2. After you rebuild `.venv`, for example by deleting it or by switching Python versions (section 12). The startup file lived inside the old `.venv`.
3. After you move or rename the `ML_Lab` folder. `uv.toml` still holds the old full path, so uv would keep using (and recreating) the cache at the old location.

Restart any open notebook kernels afterwards.

### Freeing space

Models are the big files. This lists what the Hugging Face cache holds (nothing until module 04), then the size of each folder inside `ML_Lab` that downloads land in.

In [ ]:
hub = ROOT / ".cache" / "huggingface" / "hub"
if hub.exists():
    !uv run hf cache ls
else:
    print("No Hugging Face downloads yet. Module 04 is the first to fill this folder.")

!du -sh ../../.cache/* ../../.venv 2>/dev/null

To remove one model or dataset, pass its id from the list to `hf cache rm`, in iTerm2 from `ML_Lab`:

```bash
uv run hf cache rm model/Qwen/Qwen2.5-1.5B-Instruct
```

Use `hf cache rm` rather than deleting files by hand inside `.cache/huggingface/hub`. The repos there link into one shared blob store, so deleting a repo's folder yourself can leave its files behind in `hub/blobs` and free no space. The command asks for confirmation before it deletes anything, and adding `--dry-run` previews what it would remove.

To clear a whole category, delete its folder under `ML_Lab/.cache`, for example `.cache/huggingface` or `.cache/wandb`. Nothing breaks. The next notebook that needs a model downloads it again. `uv cache clean` does the same for uv's cache, and your installed environment keeps working. Don't delete `.venv` from a notebook (section 6 explains why).

## 9. Run tools without installing them

Some command-line tools you need once, or want available everywhere without adding them to every project. `uvx <tool>` (short for `uv tool run`) installs the tool into a cached throwaway environment and runs it. Your project never knows. For a tool you use daily across projects, `uv tool install <tool>` puts it on your `PATH` for good.

One catch in this repo: uv treats `uvx` and the `uv tool` commands as user-level and ignores `ML_Lab/uv.toml` for them, so by default their downloads and throwaway environments would go to uv's cache in your home folder. The cell below passes `--cache-dir` to keep this tool inside `ML_Lab/.cache/uv` (section 8). `uv tool install` also puts tools in uv's tools folder in your home directory (`uv tool dir`). The course never uses it.

In [ ]:
before = [(ROOT / name).read_text() for name in ("pyproject.toml", "uv.lock")]
# uvx ignores ML_Lab/uv.toml (see the text above), so point it at the project's cache by hand.
!uvx --cache-dir "{ROOT}/.cache/uv" pycowsay "uvx runs tools in a throwaway env"
after = [(ROOT / name).read_text() for name in ("pyproject.toml", "uv.lock")]
print("pyproject.toml changed:", before[0] != after[0])
print("uv.lock changed:       ", before[1] != after[1])

`pycowsay` downloaded, ran, and left `pyproject.toml` and `uv.lock` alone. Run the cell again and it starts faster, because the tool's environment is now cached in `ML_Lab/.cache/uv`. (pycowsay is an old joke package, so Python 3.14 may print a `SyntaxWarning` about its code the first time. It's harmless.)

## 10. Check your hardware stack

Every later module runs models on your Mac's GPU through one of two stacks. PyTorch reaches the GPU through MPS (Metal Performance Shaders), Apple's GPU backend for PyTorch. MLX is Apple's own array framework, built for Apple Silicon's unified memory: the CPU and GPU share one pool of RAM, so arrays never get copied between them. Check that both work now, before a training run depends on them.

In [ ]:
import torch

print("PyTorch", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

# The same device pick every later notebook uses: NVIDIA GPU, then Apple GPU, then CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

x = torch.tensor([1.0, 2.0, 3.0], device=device)
print("Device:", device, " x * 2 =", x * 2)

In [ ]:
import mlx.core as mx

print("MLX", mx.__version__, " default device:", mx.default_device())
print(mx.array([1, 2, 3]) * 2)

device_info = getattr(mx, "device_info", None) or mx.metal.device_info   # newer MLX moved it to the top level
info = device_info()
gib = 2**30
print(f"{info['device_name']}: {info['memory_size'] / gib:.0f} GB unified memory, "
      f"GPU budget {info['max_recommended_working_set_size'] / gib:.1f} GB")

What to notice:

- `MPS available: True` and a tensor on `mps` mean PyTorch can train on your GPU. `cuda` only shows up on an NVIDIA machine, such as a rented GPU.
- MLX's default device is the GPU. MLX is lazy: `mx.array([1, 2, 3]) * 2` only records the operation, and the math runs when you need the result (printing counts).
- The GPU budget line is the number that matters for every model you run locally. macOS lets the GPU use only part of unified memory by default (about 17.8 GB of your 24 GB), and model weights plus the KV cache (the memory a model uses to remember the conversation so far) must fit inside it. Module 01 turns this into memory math for real models.

## 11. Commit your work

Always commit `pyproject.toml` and `uv.lock` together. One without the other means the recipe and the receipt disagree, and anyone who runs `uv sync --locked` (what a CI job or a careful teammate runs) gets an error instead of your environment.

Check what changed first:

In [ ]:
!git -C ../.. status --short

You should see `pyproject.toml` and `uv.lock` modified. `practice_app/`, `.cache/`, and `uv.toml` shouldn't appear, because they're all in `.gitignore`. If this notebook shows up as modified, that's only its saved outputs from your run. Leave it out of this commit.

The commit cell is guarded by `COMMIT = False`, so running the notebook top to bottom never commits or pushes by accident. Set it to `True`, run the cell once, then set it back. (The same commands work in iTerm2 from `ML_Lab` without the `-C ../..` part, which tells git to act on the folder two levels up.)

In [ ]:
COMMIT = False   # set to True, run this cell once, then set it back

if COMMIT:
    !git -C ../.. add pyproject.toml uv.lock
    !git -C ../.. commit -m "Finish module 00: add openai, matplotlib, ruff"
    !git -C ../.. push
else:
    print("COMMIT is False, so nothing was committed or pushed.")

## 12. If a package won't install

ML_Lab uses Python 3.14, which is very new. Most of the ML stack supports it, but occasionally a package has no prebuilt wheel for the newest Python yet. uv then tries to build it from source, and the install fails, usually with a long compiler error. If that happens, switch the project to 3.12.

Do it from iTerm2, not from a notebook, because `uv sync` rebuilds the `.venv` the kernel runs from:

1. Shut down the notebook kernel (or quit VS Code).
2. In `pyproject.toml`, change `requires-python = ">=3.14"` to `requires-python = ">=3.12"`. This comes first because uv won't pin a Python version the project says it doesn't support.
3. In iTerm2:

```bash
cd ~/Developer/ML_Lab
uv python pin 3.12
uv sync
uv run python -m ml_lab setup
```

The last line is there because switching Python versions rebuilds `.venv`, and the startup file from section 8 lives inside it.

4. Reopen VS Code and reselect the `.venv` kernel. Its Python version changed, so VS Code may ask again.

uv downloads Python 3.12 for you if you don't have it. Exercise 2 rehearses the whole switch on practice_app, where nothing can break.

## Wrap-up

Why this matters: when you hire your first engineer, onboarding to your codebase should be `git clone` and `uv sync`, and they're running the exact environment you are, down to the patch version of every package. The same lock file makes a Docker image or a CI run match your laptop. Reproducible environments stop "works on my machine" bugs, which cost small teams days they don't have.

You're done when `pyproject.toml` lists `openai` and `matplotlib` (with `ruff` under `[dependency-groups]`), section 8's check cell says your downloads land `inside ML_Lab`, section 10 prints `MPS available: True` and an MLX array, and you've committed `pyproject.toml` and `uv.lock` together.

Going further: uv's documentation at docs.astral.sh/uv covers workspaces, Docker images, and publishing packages.

---
# Exercises

These run against `practice_app/` or only read ML_Lab, so nothing here can break the course. If you've deleted `practice_app/`, rerun section 6 first. Write your attempts in new cells below this one, using `!cd practice_app && uv ...` lines like section 6.

**Exercise 1: A script with its own dependencies.** Write `practice_app/hello_table.py`, a short script that prints a table with `rich` (`from rich.table import Table`). Instead of adding `rich` to practice_app, give the script its own dependency list with `uv add --script hello_table.py rich`, then run it with `uv run hello_table.py`. Open the file and find the block uv added at the top, and confirm practice_app's `pyproject.toml` didn't change. This is inline script metadata (PEP 723): a single file that carries its own requirements, handy for a one-off script you send to someone.

**Exercise 2: Pin practice_app to Python 3.12.** Rehearse section 12 where nothing can break. First run `uv python pin 3.12` in practice_app without changing anything else and read what uv says. Then lower `requires-python` in `practice_app/pyproject.toml` to `">=3.12"`, pin again, run `uv sync`, and prove with `uv run python --version` that practice_app now runs 3.12 while ML_Lab still runs 3.14.

**Exercise 3: Constraints and upgrades.** In practice_app, add an exact old version: `uv add "rich==13.7.1"`. Then loosen the constraint in `pyproject.toml` by hand to `"rich>=13"` and run `uv lock`. Which version is locked now, and why? Then run `uv lock --upgrade-package rich` and check again. Read the locked version with `uv tree --depth 1`.

**Exercise 4: Why is this package here?** Without changing ML_Lab, find out which of your direct dependencies pull in `sympy` and which pull in `numpy`. Use `uv tree --invert --package <name>` from this folder. Which of the two would disappear from `uv.lock` if you removed `torch`?

**Exercise 5: Catch an out-of-date lock.** CI jobs and Docker builds run `uv sync --locked`, so a `pyproject.toml` change that nobody locked fails loudly instead of quietly resolving something new on a server. In practice_app, add `"tqdm>=4"` to `dependencies` by editing `pyproject.toml` by hand (no `uv add`), then run `uv sync --locked`. Read the error, fix it with `uv lock`, and run `uv sync --locked` again.

---
# Solutions

In [ ]:
# Exercise 1
script = PRACTICE / "hello_table.py"
script.write_text(
    "from rich.console import Console\n"
    "from rich.table import Table\n"
    "\n"
    'table = Table(title="uv commands")\n'
    'table.add_column("Command")\n'
    'table.add_column("What it does")\n'
    'table.add_row("uv add", "add a dependency")\n'
    'table.add_row("uv sync", "make .venv match uv.lock")\n'
    "Console().print(table)\n"
)
before = (PRACTICE / "pyproject.toml").read_text()

!cd practice_app && uv add --script hello_table.py rich
!cd practice_app && uv run hello_table.py

print(script.read_text().split("from rich")[0])   # the inline metadata block uv wrote
print("practice_app/pyproject.toml changed:", before != (PRACTICE / "pyproject.toml").read_text())

In [ ]:
# Exercise 2
pyproject_path = PRACTICE / "pyproject.toml"

# Before the change: uv refuses, because 3.12 is outside requires-python (>=3.14).
!cd practice_app && uv python pin 3.12

pyproject_path.write_text(
    pyproject_path.read_text().replace('requires-python = ">=3.14"', 'requires-python = ">=3.12"')
)
!cd practice_app && uv python pin 3.12
!cd practice_app && uv sync
!cd practice_app && uv run python --version
!uv run python --version

In [ ]:
# Exercise 3
pyproject_path = PRACTICE / "pyproject.toml"

!cd practice_app && uv add "rich==13.7.1"
!cd practice_app && uv tree --depth 1

pyproject_path.write_text(pyproject_path.read_text().replace('"rich==13.7.1"', '"rich>=13"'))
!cd practice_app && uv lock
!cd practice_app && uv tree --depth 1
# Still 13.7.1: it satisfies the new constraint, and uv keeps locked versions until you ask for an upgrade.

!cd practice_app && uv lock --upgrade-package rich
!cd practice_app && uv tree --depth 1
# uv lock only rewrites uv.lock. The next uv sync (or uv run) installs the new version into .venv.

In [ ]:
# Exercise 4
!uv tree --invert --package sympy
!uv tree --invert --package numpy
# sympy comes only from torch, so it (and its dependency mpmath) would leave the lock with torch.
# numpy stays: matplotlib (directly and through contourpy) and mlx-lm (directly and through transformers) need it too.

In [ ]:
# Exercise 5
pyproject_path = PRACTICE / "pyproject.toml"
text = pyproject_path.read_text()
if '"tqdm>=4"' not in text:
    pyproject_path.write_text(text.replace("dependencies = [", 'dependencies = [\n    "tqdm>=4",', 1))

!cd practice_app && uv sync --locked
!cd practice_app && uv lock
!cd practice_app && uv sync --locked